In [1]:
! gdown 1-AlW7oNJHaqi3xk_9dWHUS52Dzl_FmFW
! gdown 1-8TsrqTRFP-q9TM-6HinhO0ZVXFHq9TB
! gdown 1I9aPAvvYgQWdHGKtnd7IeTGXpx8vOm4h

Downloading...
From (original): https://drive.google.com/uc?id=1-AlW7oNJHaqi3xk_9dWHUS52Dzl_FmFW
From (redirected): https://drive.google.com/uc?id=1-AlW7oNJHaqi3xk_9dWHUS52Dzl_FmFW&confirm=t&uuid=e0644587-6406-44c4-91c7-95039783e0e9
To: /content/train_data.csv
100% 635M/635M [00:16<00:00, 38.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-8TsrqTRFP-q9TM-6HinhO0ZVXFHq9TB
To: /content/test_data.csv
100% 15.6M/15.6M [00:00<00:00, 99.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1I9aPAvvYgQWdHGKtnd7IeTGXpx8vOm4h
To: /content/title_brand.csv
100% 97.3M/97.3M [00:01<00:00, 85.4MB/s]


In [2]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")
product_df = pd.read_csv("title_brand.csv")

/tmp/ipykernel_2921/2058225489.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("train_data.csv")


In [ ]:
print("Train:", train_df.shape)
print("Test :", test_df.shape)
print("Products:", product_df.shape)

train_df.head()

Train: (838944, 11)
Test : (20000, 10)
Products: (786445, 3)


,overall,vote,verified,reviewTime,reviewerID,asin,style,reviewerName,reviewText,summary,unixReviewTime
0,2,NaN,False,2016-11-11,A2OSUEZJIN7BI,0511189877,NaN,Chris,I have an older URC-WR7 remote and thought thi...,Cannot Learn,1478822400
1,5,NaN,True,2016-06-06,A2NETQRG6JHIG7,0511189877,NaN,Qrysta White,First time I've EVER had a remote that needed ...,zero programming needed! Miracle!?,1465171200
2,4,NaN,True,2016-03-10,A12JHGROAX49G7,0511189877,NaN,Linwood,Got them and only 2 of them worked. company ca...,Works Good and programs easy.,1457568000
3,5,NaN,True,2016-01-14,A1KV65E2TMMG6F,0511189877,NaN,Dane Williams,I got tired of the remote being on the wrong s...,Same as TWC remote,1452729600
4,5,NaN,True,2016-10-20,A280POPEWI0NSA,0594459451,NaN,Kristina H.,After purchasing cheap cords from another webs...,Good Quality Cord,1476921600


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 838944 entries, 0 to 838943
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   overall         838944 non-null  int64 
 1   vote            191468 non-null  object
 2   verified        838944 non-null  bool  
 3   reviewTime      838944 non-null  object
 4   reviewerID      838944 non-null  object
 5   asin            838944 non-null  object
 6   style           490613 non-null  object
 7   reviewerName    838717 non-null  object
 8   reviewText      838944 non-null  object
 9   summary         838868 non-null  object
 10  unixReviewTime  838944 non-null  int64 
dtypes: bool(1), int64(2), object(8)
memory usage: 64.8+ MB


In [ ]:
train_df.isnull().sum()

,0
overall,0
vote,647476
verified,0
reviewTime,0
reviewerID,0
asin,0
style,348331
reviewerName,227
reviewText,0
summary,76


<div dir="rtl">
    <h2>میزان رضایت از یک جنبه‌ی مشخص</h2>
</div>

In [4]:
! pip install gensim
! pip install RapidFuzz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.9 MB/s eta 0:00:00


In [5]:
import re
import numpy as np
import pandas as pd

from gensim.models import Word2Vec
from rapidfuzz import process, fuzz

In [ ]:
def tokenize_text(text):
    text = str(text).lower()

    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text.split()

train_df["tokens"] = train_df["reviewText"].fillna("").apply(tokenize_text)

In [ ]:
sample_size = min(100000, len(train_df))

sample_df = train_df.sample(
    n=sample_size,
    random_state=42
)

sentences = sample_df["tokens"].tolist()

word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    sg=1,
    epochs=5,
    seed=42
)

In [9]:
warranty_similar = word2vec_model.wv.most_similar(
    "warranty",
    topn=30
)

guarantee_similar = word2vec_model.wv.most_similar(
    "guarantee",
    topn=30
)
print("Similar words to WARRANTY:")
print("-" * 50)

for word, score in warranty_similar:
    print(f"{word:<20} {score:.4f}")


print("\nSimilar words to GUARANTEE:")
print("-" * 50)

for word, score in guarantee_similar:
    print(f"{word:<20} {score:.4f}")

Similar words to WARRANTY:
--------------------------------------------------
expired              0.7904
warrenty             0.7737
warrantee            0.7729
warranties           0.7651
expires              0.7557
honor                0.7537
honored              0.7103
lifetime             0.7035
deadline             0.6996
honoring             0.6916
policy               0.6877
expiration           0.6869
rebate               0.6712
guarantee            0.6679
expire               0.6637
returnable           0.6600
turnaround           0.6587
asurion              0.6574
warrantied           0.6522
voided               0.6522
repairs              0.6501
squaretrade          0.6499
reimbursement        0.6499
repairable           0.6434
eligible             0.6380
manufacturer         0.6376
refunds              0.6343
authorization        0.6276
drevo                0.6268
quoted               0.6261

Similar words to GUARANTEE:
--------------------------------------------------
ho

In [10]:
semantic_candidates = {}

for word, score in warranty_similar:
    semantic_candidates[word] = score

for word, score in guarantee_similar:
    if word not in semantic_candidates:
        semantic_candidates[word] = score
    else:
        semantic_candidates[word] = max(
            semantic_candidates[word],
            score
        )
semantic_candidates = dict(
    sorted(
        semantic_candidates.items(),
        key=lambda x: x[1],
        reverse=True
    )
)
for word, score in semantic_candidates.items():
    print(f"{word:<20} {score:.4f}")

expired              0.7904
warrenty             0.7737
warrantee            0.7729
warranties           0.7651
expires              0.7557
honor                0.7537
promise              0.7164
honored              0.7103
lifetime             0.7035
deadline             0.6996
honoring             0.6916
policy               0.6877
expiration           0.6869
affiliation          0.6866
commitment           0.6813
rebate               0.6712
warranty             0.6679
guarantee            0.6679
expire               0.6637
instill              0.6624
returnable           0.6600
turnaround           0.6587
asurion              0.6574
warrantied           0.6522
voided               0.6522
endorse              0.6514
repairs              0.6501
squaretrade          0.6499
reimbursement        0.6499
refunds              0.6469
repairable           0.6434
agreement            0.6397
eligible             0.6380
manufacturer         0.6376
deception            0.6362
judgment            

In [ ]:
from rapidfuzz import process, fuzz

strong_terms = {"warranty","warranties","warrenty","warrantee","warrantied","guarantee","guarantees"}
related_terms = {"expired","expires","expire","expiration",
    "voided","repairs","repairable","reimbursement","refunds","asurion",
    "squaretrade","manufacturer","coverage","replacement","replace","replaced"
}

def is_fuzzy_match(word, target_terms, threshold=88):

    if word in target_terms:
        return True

    result = process.extractOne(word,list(target_terms), scorer=fuzz.ratio)

    if result is None:
        return False

    matched_word, score, _ = result

    return score >= threshold

In [ ]:
def contains_warranty_term(tokens):

    for word in tokens:
        if is_fuzzy_match(word,strong_terms, threshold=88):
            return True

    for i, word in enumerate(tokens):
        if word not in related_terms:
            continue

        start = max(0, i - 7)
        end = min(len(tokens), i + 8)
        context = tokens[start:end]

        if any(is_fuzzy_match(w,strong_terms,threshold=88)for w in context):
            return True

    return False

In [ ]:
train_df["has_warranty"] = train_df["tokens"].apply(
    contains_warranty_term
)

print(train_df["has_warranty"].value_counts())
warranty_reviews = train_df[train_df["has_warranty"]].copy()

print("Total reviews:", len(train_df))
print("Warranty-related reviews:", len(warranty_reviews))

print(f"Percentage: "f"{len(warranty_reviews) / len(train_df) * 100:.2f}%")

has_warranty
False    819868
True      19076
Name: count, dtype: int64
Total reviews: 838944
Warranty-related reviews: 19076
Percentage: 2.27%


In [ ]:
pd.set_option("display.max_colwidth", 500)

sample_reviews = warranty_reviews[["asin", "overall", "reviewText"]].sample(
    min(30, len(warranty_reviews)),
    random_state=42
)

sample_reviews

,asin,overall,reviewText
529506,B014M8ZO92,2,"Although this wasn't an original feature of UE Boom 2, they added integration to Google Now / Siri through an update, and this feature tipped me toward this $200 purchase over competing devices.\n\nUnfortunately - it didn't work! Not on my Android phone, nor with the 2 other phones I tested. This feature DOES work using my phone with other bluetooth audio devices I own (two headsets with a Google Now button and a car with bluetooth button.)\n\nCustomer support started off poorly, with no res..."
712980,B01G1NHLT2,1,"I have gone through three pair of Jabra's Halo Smart IIs. It's too bad because I find them to be the best balance of sound quality, weight on the neck so it doesn't fall off and down your back, and battery life is excellent. Unfortunately, my first pair shorted out in one ear after the warranty ended, the second pair just decided to stop re-charging one day, and the third kind of fell apart because of the plastic design. If they lowered the price to $19 I'd buy 4 and deal with it. There s..."
295709,B00L23XFNS,2,".Comfortable, good fit, good sound. Wish the buttons had more tactile differentiation.\n\nAfter about 1 year, 3 months (3 days/week at the gym), the charger port broke. The connector inside the headphones pulled out when unplugging the USB cable. So now it's useless, and 3 months out of warranty."
340028,B00NW2JBYO,1,"We had this for 2 months - the part cracked at an assembly point under normal use ; returned it to warranty center ; it was promptly sent back with no explanations -- once we received ; found the unit broken just as we had sent it -- along with some heat damage along with more portions broken ( I took pictures before we sent ) - I got similar answers with person I talked to on phone and with LG Live chat\nInput from LG "" I have checked the RA#. The headset was returned to you as it is bec..."
359757,B00PEPIWNI,5,"As a long-story-short, I bought these back in February of 2016 and they were magical for 33 days before breaking spontaneously.\nI wrote a sour review because they were 3 days past the Amazon return/exchange period and that really sucked.\nBut lets make something clear- These are AWESOME, feature packed headphones for $20. My original pair sounded better than any other pair of <$100 headphones, in ear or otherwise, that I have ever owned. My 33 days with them were good enough that I wrote my..."
830365,B01CZM679I,1,"Works like crap out of the box! you'll be constantly re-minimizing the drive space windows 10 - windows updates are always an ordeal - always dealing with low memory. Will run slow if you don't void warranty by taking battery out of device.\n\nMy work bought 40 of these for workstations, it took about 4 hours each and voided warranties to set these up to run for single purpose workstations. Each one has different issues - some have wireless card regularly disabling itself even with updated..."
356371,B00P7TQR06,1,"If you have a iPad mini 2 o r iPad mini 3 this case is certainly worth considering, though I bought mine and a key fell off after 2 months and Zagg warranty service is lacking. I have used this unit daily for those 2 months and it worked flawlessly. Being able to convert from a laptop function, or a tablet and also being able to leave the keyboard in a bag, was invaluable and extended the function of my iPad.\n\nZagg has offered to warranty replace, but their warranty replacement does not ..."
203621,B00E64BE6A,4,Had to use the warranty on a Danby microwave. Good to deal with and check for defective unit arrived quickly. The down side for us was the requirement for my to dispose of the old microwave. Cost $5 to dispose of.
789729,B00N20BAVS,1,"It worked great on my Macbook Air for about less than a day. Then the ethernet connect failed. I was able to reboot and reestablish, but an hour later it failed again. Overheating?\n\nThe USB ports still work ok, but I bought this primarily for the ethernet functionality. The 

In [ ]:
rating_distribution = warranty_reviews["overall"].value_counts().sort_index()

print(rating_distribution)
rating_percentage = warranty_reviews["overall"].value_counts(normalize=True).sort_index()* 100

print(rating_percentage.round(2))

overall
1    5236
2    2097
3    1966
4    2587
5    7190
Name: count, dtype: int64
overall
1    27.45
2    10.99
3    10.31
4    13.56
5    37.69
Name: proportion, dtype: float64


In [ ]:
overall_warranty_mean = warranty_reviews["overall"].mean()

print(f"Average rating for warranty-related reviews: "f"{overall_warranty_mean:.3f} / 5")

Average rating for warranty-related reviews: 3.231 / 5


In [ ]:
product_warranty_stats = warranty_reviews.groupby("asin").agg(mean_overall=("overall", "mean"),review_count=("overall", "count")).reset_index()

product_warranty_stats.head(20)

,asin,mean_overall,review_count
0,6541654530,1.000000,1
1,9800466657,5.000000,1
2,B000001OM4,4.000000,1
3,B00000J4EY,3.000000,1
4,B00000K2YR,2.333333,3
5,B00001P4XA,5.000000,2
6,B00001P4ZH,4.352941,17
7,B00001P505,5.000000,2
8,B000026D8E,1.000000,1
9,B00002EQCS,3.000000,2


In [31]:
product_warranty_stats = product_warranty_stats.sort_values(
    by="mean_overall",
    ascending=False
)

product_warranty_stats.head(20)

,asin,mean_overall,review_count
1,9800466657,5.0,1
5,B00001P4XA,5.0,2
10350,B01HG0TYGM,5.0,1
10349,B01HFFB89Y,5.0,1
10348,B01HEXFQVI,5.0,1
10347,B01HEV625Y,5.0,1
10346,B01HEO5J5U,5.0,1
7,B00001P505,5.0,2
10365,B01HIURQWE,5.0,1
10366,B01HIWBU7Y,5.0,1


In [ ]:
reliable_products = product_warranty_stats[
    product_warranty_stats["review_count"] >= 10
].copy()

print(f"Products before filtering: "f"{len(product_warranty_stats):,}")

print(f"Products with at least 10 reviews: "f"{len(reliable_products):,}")

Products before filtering: 10,368
Products with at least 10 reviews: 182


In [33]:
top_20_products = (
    reliable_products
    .sort_values(
        "mean_overall",
        ascending=False
    )
    .head(20)
)

top_20_products

,asin,mean_overall,review_count
4552,B00N1O1NQW,5.000000,29
8444,B0192GJLMU,4.857143,14
5431,B00SF6VSVG,4.818182,11
6879,B011KOQ8HS,4.764706,17
205,B0009WH4OE,4.750000,12
298,B000JE7GPY,4.750000,16
9583,B01DZ56MX0,4.692308,13
3123,B00FX6KO8Y,4.636364,11
8701,B01A6PGZ70,4.625000,16
28,B0000510R4,4.600000,10


In [34]:
bottom_20_products = (
    reliable_products
    .sort_values(
        "mean_overall",
        ascending=True
    )
    .head(20)
)

bottom_20_products

,asin,mean_overall,review_count
7041,B012PKV7RC,1.090909,11
2968,B00F0DD0I6,1.300000,10
8963,B01BHLK0MS,1.333333,12
3209,B00GKKI4IE,1.375000,16
3105,B00FRHTTJE,1.437500,16
6521,B00YZF7PEU,1.684211,19
6637,B00ZSPXR8E,1.684211,19
7679,B015PD3HOC,1.687500,16
6351,B00XWPAJ1U,1.700000,10
5484,B00SMBG8QY,1.700000,10


In [35]:
final_warranty_results = (
    reliable_products[
        ["asin", "mean_overall", "review_count"]
    ]
    .sort_values(
        "mean_overall",
        ascending=False
    )
    .reset_index(drop=True)
)

final_warranty_results.head(30)

,asin,mean_overall,review_count
0,B00N1O1NQW,5.000000,29
1,B0192GJLMU,4.857143,14
2,B00SF6VSVG,4.818182,11
3,B011KOQ8HS,4.764706,17
4,B0009WH4OE,4.750000,12
5,B000JE7GPY,4.750000,16
6,B01DZ56MX0,4.692308,13
7,B00FX6KO8Y,4.636364,11
8,B01A6PGZ70,4.625000,16
9,B0000510R4,4.600000,10


In [ ]:
print(f"Total reviews: {len(train_df):,}")
print(f"Warranty-related reviews: "f"{len(warranty_reviews):,}")
print(f"Warranty-related percentage: "f"{len(warranty_reviews) / len(train_df) * 100:.2f}%")
print(f"Overall warranty satisfaction: "f"{overall_warranty_mean:.3f} / 5")
print(f"Products analyzed (>=10 reviews): "f"{len(reliable_products):,}")

FINAL WARRANTY SATISFACTION RESULTS
Total reviews: 838,944
Warranty-related reviews: 19,076
Warranty-related percentage: 2.27%
Overall warranty satisfaction: 3.231 / 5
Products analyzed (>=10 reviews): 182
